# CLPsych 2026 — Experiment Runner

In [ ]:
# 1. Setup
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader  # optional: confirms a GPU is available
%pip install -q -r requirements.txt

In [ ]:
# 2. Experiment config — change these for each experiment
MODEL = 'Qwen/Qwen3.5-27B'
TEMPERATURE = 0.1
MAX_TOKENS = 100
PER_ELEMENT_A = True  # separate call for element A

# RAG retrieval tuning
RAG_K = 8
RAG_POOL = 20     # candidate pool size (default 20, try 100-200)
RAG_ALPHA = 0.6    # similarity vs diversity weight (default 0.6, try 0.4)

In [ ]:
# 3. Load data & RAG indexes
from data_loader import load_all_timelines
from rag_index import RAGIndex

# Real data — used for unified RAG + evaluation
posts = load_all_timelines('train_tasks12')
evidenced = [p for p in posts if p.has_evidence]
print(f'Evidenced (real): {len(evidenced)}')

# Augmented data — used for per-element A RAG only
aug_posts = load_all_timelines('augmented_data')
aug_evidenced = [p for p in aug_posts if p.has_evidence]
print(f'Augmented: {len(aug_evidenced)}')

# Unified RAG: real data only
rag_index_real = RAGIndex(evidenced)
print(f'RAG real: {len(rag_index_real.posts)} posts')

# Per-element A RAG: real + augmented
rag_index_aug = RAGIndex(evidenced + aug_evidenced)
print(f'RAG augmented: {len(rag_index_aug.posts)} posts')

In [ ]:
# 4. Build prompts (calls 1-3)
from prompt_builder_rag import build_messages, build_messages_adaptive, build_messages_per_element
from tqdm import tqdm

target = evidenced  # use evidenced[:5] for quick test

# Call 1: unified prompts
all_messages = []
rag_cache = []
for p in tqdm(target, desc='Unified'):
    rag = rag_index_real.retrieve(p.text, k=RAG_K, exclude_timeline=p.timeline_id,
                                  candidate_pool=RAG_POOL, diversity_alpha=RAG_ALPHA)
    all_messages.append(build_messages(p.text, rag))
    rag_cache.append(rag)

# Call 3: per-element A prompts
messages_A = []
for p in tqdm(target, desc='Element A'):
    rag_A = rag_index_aug.retrieve_for_element(p.text, 'A', k=RAG_K, exclude_timeline=p.timeline_id)
    messages_A.append(build_messages_per_element(p.text, 'A', rag_A))

print(f'Prompts: {len(all_messages)} unified + {len(messages_A)} element A')

In [ ]:
# 5. Load model
from vllm import LLM, SamplingParams

llm = LLM(MODEL, dtype='float16', max_model_len=8192)
params = SamplingParams(max_tokens=MAX_TOKENS, temperature=TEMPERATURE)
print('Model ready!')

In [ ]:
# 6. Inference: call 1 (unified) → call 2 (adaptive) → call 3 (per-element A)
from output_parser import parse_llm_output

# Call 1: unified
print('Call 1: unified ...')
outputs = llm.chat(all_messages, sampling_params=params, chat_template_kwargs={'enable_thinking': False})
print(f'Done: {outputs[0].outputs[0].text}')

# Parse unified to get maladaptive context for call 2
results_unified = [parse_llm_output(o.outputs[0].text.strip()) for o in outputs]

# Build adaptive-focused prompts with maladaptive context
messages_adaptive = []
for i, (p, u_pred) in enumerate(zip(target, results_unified)):
    messages_adaptive.append(build_messages_adaptive(p.text, rag_cache[i], u_pred['maladaptive']))

# Call 2: adaptive-focused
print('Call 2: adaptive-focused ...')
outputs_adaptive = llm.chat(messages_adaptive, sampling_params=params, chat_template_kwargs={'enable_thinking': False})
print(f'Done: {outputs_adaptive[0].outputs[0].text}')

# Call 3: per-element A
print('Call 3: per-element A ...')
outputs_A = llm.chat(messages_A, sampling_params=params, chat_template_kwargs={'enable_thinking': False})
print(f'Done: {outputs_A[0].outputs[0].text}')

In [ ]:
# 7. Merge & evaluate
# Maladaptive: from unified (call 1), A from per-element A (call 3)
# Adaptive: elements from adaptive-focused (call 2), presence from unified (call 1)
# A (both valences): from per-element A (call 3)
from evaluate import evaluate_task11, evaluate_task12, print_evaluation, log_experiment
from submission_formatter import save_submission
from datetime import datetime
import json
from pathlib import Path

results = []
for i, post in enumerate(target):
    u_pred = results_unified[i]
    a_pred = parse_llm_output(outputs_adaptive[i].outputs[0].text.strip())
    a_elem_pred = parse_llm_output(outputs_A[i].outputs[0].text.strip())

    merged = {
        'maladaptive': {
            'elements': dict(u_pred['maladaptive']['elements']),
            'presence': u_pred['maladaptive']['presence'],
        },
        'adaptive': {
            'elements': dict(a_pred['adaptive']['elements']),
            'presence': u_pred['adaptive']['presence'],
        },
    }
    # Override A from per-element A (call 3) for both valences
    for v in ('adaptive', 'maladaptive'):
        merged[v]['elements'].pop('A', None)
        if 'A' in a_elem_pred[v]['elements']:
            merged[v]['elements']['A'] = a_elem_pred[v]['elements']['A']

    # Consistency: no elements → presence = 1
    for v in ('adaptive', 'maladaptive'):
        if not merged[v]['elements']:
            merged[v]['presence'] = 1

    results.append({
        'timeline_id': post.timeline_id,
        'post_id': post.post_id,
        'raw_output': outputs[i].outputs[0].text.strip(),
        'prediction': merged,
    })

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
raw_path = f'outputs/raw_results_twopass_perA_{ts}.json'

t11 = evaluate_task11(target, results)
t12 = evaluate_task12(target, results)
print_evaluation(t11, t12)

notes = input('Experiment notes: ')
log_experiment(t11, t12, config={
    'model': MODEL, 'rag_k': RAG_K,
    'temperature': TEMPERATURE, 'max_tokens': MAX_TOKENS,
    'num_posts': len(target), 'raw_results_file': raw_path,
}, notes=notes)

In [ ]:
# 8. Save results
Path('outputs').mkdir(exist_ok=True)
with open(raw_path, 'w') as f:
    json.dump(results, f, indent=2)
save_submission(results, raw_path.replace('raw_results', 'task1_pred'))


In [ ]:
# 9. Validate submission & official evaluation
import sys, subprocess, shutil
if 'official_eval' not in sys.path:
    sys.path.insert(0, 'official_eval')

sub_path = raw_path.replace('raw_results', 'task1_pred')

# Validate submission format (validator requires filename = task1_pred.json)
print('=== Submission Validation ===')
shutil.copy(sub_path, 'outputs/task1_pred.json')
result = subprocess.run(
    [sys.executable, 'official_eval/validate_submission.py',
     '--task1', 'outputs/task1_pred.json', '--test-dir', 'train_tasks12'],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    print('WARNING: validation failed, check submission format')

# Run official evaluation (no v2 conversion needed — organizers fixed the eval script)
print('\n=== Official Evaluation ===')
from evaluate_task1 import evaluate as official_evaluate
official_results = official_evaluate('train_tasks12', sub_path)